In [ ]:
## Packages Import
%matplotlib widget
import copy
import time
import numpy             as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

In [ ]:
class VLMPanel:
    """
    Vortex Lattice Method panel with ring vortex
    """
    #--------------------#
    #   Initialization   #
    #--------------------#
    def __init__(self, p, v, w, b):
        """
        Panel is of quadrilateral shape and vertices must be ordered from root
        position on leading-edge proceeding in counterclockwise direction.
        The pannel can be a wing pannel (with an offset ring) or a wake pannel (with the ring being the pannel)
        """
        # Wing geometry
        self.span = b       # wing span [m]

        # Wing or wake
        self.nature = w        # 0 for wing, 1 for wake

        # Panel geometry
        self.pnt    = p        # panel vertices
        self.ctr    = None     # control point
        self.normal = None     # normal direction
        self.chord  = None     # chord length [m]
        self.width  = None     # pannel width [m]
        self.area   = None     # panel area   [m**2]
        
        # Vortex parameters
        self.vrt = v            # ring vortex points
        self.bound = None       # length of the bound segment in the y axis


        self._get_panel_geom()
      


    #-------------#
    #   Methods   #
    #-------------#
    def _get_panel_geom(self):
        """
        Compute panel geometric parameters such as control point position, normal
        versor, chord length, width and area, starting from panel's vertices.
        """
        p = self.pnt
        v = self.vrt
        w = self.nature
        if w == 0 :
            # Compute representative panel's geometric parameters
            c_avg = 0.5 * (np.linalg.norm(p[2] - p[1]) + np.linalg.norm(p[3] - p[0]))  # average chord length
            w_avg = 0.5 * (np.linalg.norm(p[2] - p[3]) + np.linalg.norm(p[1] - p[0]))  # average width

            # Compute position of control point
            cp = (p[0] + p[1] + 3*p[3] + 3*p[2])/8    # three-quarter line

            # Compute normal versor and panel surface
            ai = np.zeros(3)
            for i, pi in enumerate(p):
                qi = p[(i + 1) % len(p)]
                ai += np.cross(pi, qi)
            av = 0.5 * ai
            a = np.linalg.norm(av)
            if a > 0.0:
                n = av / a
            else:
                # Degenerate cells with zero area
                print(f"Degenerate panel {i} with {len(p)} vertices")

            b = v[1] - v[0]

            self.bound  = b[2]
            self.ctr    = cp
            self.normal = n
            self.chord  = c_avg
            self.width  = w_avg
            self.area   = a


In [ ]:
class VLMSolver:
    """
    Vortex Lattice Method solver
    """
    #--------------------#
    #   Initialization   #
    #--------------------#
    def __init__(self, b, c, alpha, lamb, delta, sym,
                u_inf, n, m):
        # Wing geometry
        self.span     = b       # wing span                 [m]
        self.chord    = c       # chord lengths             [m]
        self.aoa      = alpha   # angle of attack           [rad]
        self.sweep    = lamb    # quarter-chord sweep angle [rad]
        self.dihedron = delta   # dihedral angle            [rad]
        self.symmetric = sym    # symmetric wing

        # Flow properties
        self.U = u_inf      # inflow velocity [m/s]

        # Discretization
        self.N = n                  # number of panels in spanwise direction
        self.M = m                  # number of panels in chordwise direction
        self.wing_panels = None     # wing panel objects
        self.te          = None     # wing trailing edge points (ring)
        self.wake_panels = None     # wake panel objects

        # Linear system
        self.A = None       # coefficient matrix
        self.b = None       # right hand side
    
    #-------------#
    #   Methods   #
    #-------------#
    def _unit_rot(self, v, theta, ax):
        """
        Elemental clockwise rotation of the reference system axes.
        
        Input:
        v     -> vector componets - shape: (N, 3)
        theta -> rotation angle - clockwise rotation: theta>0
        ax    -> rotation axis index
        """
        # Assemble rotation matrix
        rot = np.zeros((3, 3))  # rotation matrix
        # diagonal elements
        rot[ax, ax] = 1.0
        rot[ax-1, ax-1] = np.cos(theta)
        rot[ax-2, ax-2] = np.cos(theta)
        # extra-diagonal elements
        rot[ax-2, ax-1] = -np.sin(theta)
        rot[ax-1, ax-2] = np.sin(theta)

        # Apply rotation
        v_rot = np.sum(rot[None, :, :] * v[:, None, :], axis=2)
        return v_rot


    def _build_mesh(self):
        """
        Discretization of the wing by a finite number of lattices.
        """
        m = self.M
        n = self.N
        b     = self.span
        alpha = self.aoa
        lamb  = self.sweep
        delta = self.dihedron
        c     = self.chord
        # Leading-edge length of the semi-wing
        le = (0.5 * b) /  np.cos(delta)
        if self.symmetric:
            s_min, s_max = 0.0, le
        else:
            s_min, s_max = -le, l
        # Spanwise wing discretization
        s = np.linspace(s_min, s_max, (n+1))    # leading-edge coordinates
        s = np.tile(s,m+1)
        # Chordwise wing panels discretiration
        i = np.arange(m+1)[:, None]  
        j = c[None, :]                
        ch = (i/m) * j               
        ch = ch.ravel()              
        # Change panels corner points of frame
        p_wing  = np.column_stack((ch, np.zeros_like(s), s))                                          # wing frame
        p_swept = np.column_stack((p_wing[:,2]*np.tan(lamb)+p_wing[:,0],p_wing[:,1],p_wing[:,2]))     # swept frame
        p_yaw   = self._unit_rot(p_swept, -delta, 0)                                                  # yawed frame
        p_earth = self._unit_rot(p_yaw,  -alpha, 2)                                                   # earth frame
        # Ring vertices
        r_wing = copy.copy(p_wing)
        for j in range(n+1):
            for i in range(m):
                r_wing[i*(n+1)+j][0] += (r_wing[(i+1)*(n+1)+j][0]-r_wing[i*(n+1)+j][0]) * 0.25        # one quarter chord offset
            r_wing[m*(n+1)+j][0] += (r_wing[m*(n+1)+j][0]-r_wing[(m-1)*(n+1)+j][0])/3
        # Change of frame
        r_swept = np.column_stack((r_wing[:,2]*np.tan(lamb)+r_wing[:,0],r_wing[:,1],r_wing[:,2]))     # swept frame
        r_yaw   = self._unit_rot(r_swept, -delta, 0)                                                  # yawed frame
        r_earth = self._unit_rot(r_yaw,  -alpha, 2)                                                   # earth frame
        self.te = r_earth[-(n+1):]
        panels = []     # list of panel objects
        for i in range(m):
            for j in range(n):
                pnt = [p_earth[i*(1+n)+j],
                    p_earth[i*(n+1)+j+1],
                    p_earth[(i+1)*(n+1)+j+1],
                    p_earth[(i+1)*(n+1)+j]]    # ordered panel vertices    
                vtx = [r_earth[i*(1+n)+j],
                    r_earth[i*(n+1)+j+1],
                    r_earth[(i+1)*(n+1)+j+1],
                    r_earth[(i+1)*(n+1)+j]]    # ordered ring vertices
                panels.append(VLMPanel(p=pnt, v=vtx, w=0, b=b))
        self.wing_panels = panels
        self.wake_panels = []

    def _induced_velocity (self,p,c1,c2,gamma):
        """
        Return the velocity induced by the vortex segment [c1, c2], of strentgh gamma,
        at the point p
        """
        r0 = c2 - c1
        r1 = p - c1
        r2 = p - c2
        r1_r2 = np.cross(r1,r2)     #cross product of r1 and r2
        if np.linalg.norm(r1_r2) == 0 :
            return 0
        K = gamma/(4*np.pi*(np.linalg.norm(r1_r2)**2))*(np.dot(r0,r1)/np.linalg.norm(r1)-np.dot(r0,r2)/np.linalg.norm(r2))
        return r1_r2*K
    
    def _total_induced_velocity (self,p,gamma,gamma_wake):
        """
        Return the velocity induced by the wake and the wing at the point p
        """
        v = np.array([0,0,0])
        for i, ri in enumerate(self.wing_panels) :
            r = ri.pnt
            v = v + ( self._induced_velocity(p,r[0],r[1],gamma[i]) +
                self._induced_velocity(p,r[1],r[2],gamma[i]) + 
                self._induced_velocity(p,r[2],r[3],gamma[i]) + 
                self._induced_velocity(p,r[3],r[0],gamma[i]) ) 
        for i, ri in enumerate(self.wake_panels) :
            r = ri.pnt
            v = v + ( self._induced_velocity(p,r[0],r[1],gamma_wake[i]) +
                    self._induced_velocity(p,r[1],r[2],gamma_wake[i]) + 
                    self._induced_velocity(p,r[2],r[3],gamma_wake[i]) + 
                    self._induced_velocity(p,r[3],r[0],gamma_wake[i]) )
        return v

    def _build_A (self):
        """
        Construct the influence coefficients matrix A
        """
        m, n = self.M, self.N
        A = np.zeros((n*m,n*m))
        for i, ni in enumerate(self.wing_panels):
            for j, vj in enumerate(self.wing_panels):
                v = vj.vrt
                v_ring = ( self._induced_velocity(ni.ctr,v[0],v[1],1) +
                self._induced_velocity(ni.ctr,v[1],v[2],1) + 
                self._induced_velocity(ni.ctr,v[2],v[3],1) + 
                self._induced_velocity(ni.ctr,v[3],v[0],1) )

                A[i][j] = np.dot(v_ring, ni.normal)
                self.A =A

    def _build_b (self, gamma_wake):
        """ 
        Construct the RHS b

        Input :
            -> gamma_wake the vortex strentgh of each wake ring
        """
        m, n = self.M, self.N
        b = np.zeros(m*n)
        for i, ni in enumerate(self.wing_panels):
            for j, rj in enumerate(self.wake_panels):
                r = rj.vrt
                v_ring = ( self._induced_velocity(ni.ctr,r[0],r[1],gamma_wake[j]) +
                self._induced_velocity(ni.ctr,r[1],r[2],gamma_wake[j]) + 
                self._induced_velocity(ni.ctr,r[2],r[3],gamma_wake[j]) + 
                self._induced_velocity(ni.ctr,r[3],r[0],gamma_wake[j]) )
                b[i] += - np.dot(v_ring, ni.normal)
            b[i] += - np.dot(self.U, ni.normal)
        self.b = b
    
    def _lift (self, gamma):
        """
        Compute the total lift of the wing
        """
        n     = self.N
        u_inf = self.U
        L = 0
        for k, pk in enumerate(self.wing_panels):
            if k < n :          # Leading edge computation
                L += gamma[k]*pk.bound
            else :
                L += (gamma[k] - gamma[k-n])*pk.bound
        return -L*u_inf[0]
        

In [ ]:
## Utilities

def chord_fn(n, c0, b, delta, shape="rectangular"):
    """
    Return chord length distribution along spanwise direction according to the
    desired planform shape.

    Input:
        n     -> number of spanwise pannels
        c0    -> root chord length
        b     -> wing span
        delta -> dihedral angle
        shape -> wing planform
    """
    match shape:
        case "rectangular":
            c = c0 * np.ones(n)
        case "elliptical":
            le = (0.5 * b) /  np.cos(delta)
            s_min, s_max = 0.0, le
            s = np.linspace(s_min, s_max, n)
            c = c0 * np.sqrt(1 - (s/b)**2)
    return c




In [ ]:
## Parameters Definition

# Wing geometry
B = 1         # wing span                 [m]
AR = 8        # aspect ration             [-]
ALPHA  = 8   # angle of attack            [deg] - positive definite for counterclockwise rotations about z-axis
LAMBDA = 0  # quarter-chord sweep angle [deg] - positive definite for counterclockwise rotations about x-axis  Y-AXIS
DELTA  = 0  # dihedral angle            [deg] - positive definite for rotations oriented towards positive y-axis  X-AXIS, négatif pour dyhedre classique 

SYM = True   # symmetric wing configuration

# Flow properties
U = 1.0     # inflow velocity [m/s]

# Wing discretization
N = 10       # number of panels in spanwise direction
M = 2       # number of panels in chordwise direction

# Time simulation parameters
T  = 2      # length of the simulation in s
DT = 0.2       # time step lentgh 

# Untapered wing (constant chord length distribution)
c0 = B / AR     # chord length [m]



In [ ]:
## Time stepping algorithm 
def time_sim(b=B, u_inf=np.array([U, 0.0, 0.0]), n=N, t=T, dt=DT):
        """ 
        Do the time stepping simulation with the wake relaxation
        """
        # Initialization
        vlm = VLMSolver(b=B,
                c=chord_fn((N+1), c0, B, DELTA),
                alpha=np.deg2rad(ALPHA),
                lamb=np.deg2rad(LAMBDA),
                delta=np.deg2rad(DELTA),
                sym=SYM,
                u_inf=np.array([U, 0.0, 0.0]),
                n=N,m=M)
        # we memorize the wake ring strentgh
        gamma_wake = []     
        vlm._build_mesh()
        vlm._build_A()
        inv_A = np.linalg.inv(vlm.A)
        vlm._build_b(gamma_wake)
        Gamma = inv_A @ vlm.b                   
        Te  = vlm.te                    # get the trailing edge for the shedding
        LeW = Te + dt*u_inf
        t0 = time.time() 
        for s in range(round(t/dt)+1):
                # simulate the wing advancement
                for j, rj in enumerate(vlm.wake_panels) :
                        for i, p in enumerate(rj.pnt) :
                                vlm.wake_panels[j].pnt[i] = np.array([p[0]+u_inf[0]*dt,p[1]+u_inf[1]*dt,p[2]+u_inf[2]*dt])
                        if j < n :                      # needed for the shedding
                                LeW[j] = vlm.wake_panels[j].pnt[0]
                        if j == n-1 :
                                LeW[n] = vlm.wake_panels[j].pnt[1]
                # shed one row of ring
                W = []
                for j in range(n):
                        pnt = [Te[j], Te[j+1], LeW[j+1], LeW[j]]
                        W.append(VLMPanel(p=pnt, v=pnt, w=1, b=b))
                vlm.wake_panels = W + vlm.wake_panels
                gamma_wake = np.concatenate([Gamma[-(n+1):], gamma_wake])
                # update the right hand side and gamma copmutation
                vlm._build_b(gamma_wake)
                Gamma = inv_A @ vlm.b 
                # simulate the wake rollup
                W = []
                for j, r in enumerate(vlm.wake_panels) :          # awfull computation, I supposed that the speed functions will themselve optimize
                        vrt = []
                        if j < n :                     # no rollup for the corner points attached to the trailing edge rings
                                vrt.append(r.pnt[0])
                                vrt.append(r.pnt[1])
                        else :
                                vrt.append(r.pnt[0] + dt*vlm._total_induced_velocity(r.pnt[0], Gamma, gamma_wake))
                                vrt.append(r.pnt[1] + dt*vlm._total_induced_velocity(r.pnt[1], Gamma, gamma_wake))
                        vrt.append(r.pnt[2] + dt*vlm._total_induced_velocity(r.pnt[2], Gamma, gamma_wake))
                        vrt.append(r.pnt[3] + dt*vlm._total_induced_velocity(r.pnt[3], Gamma, gamma_wake))
                        W.append(VLMPanel(p=vrt, v=vrt, w=1, b=b))
                vlm.wake_panels = W
        t1 = time.time()
        # plot
        fig_3d = plt.figure(figsize=(6, 8))
        ax4 = fig_3d.add_subplot(111, projection='3d')
        for i, panel in enumerate(vlm.wing_panels):
                pnt    = copy.copy(panel.pnt)
                vrt    = copy.copy(panel.vrt)
                # Close the polygon shape
                pnt.append(pnt[0])
                pnt_plt = np.array(pnt)
                vrt.append(vrt[0])
                vrt_plt = np.array(vrt)

                # Plot lattice
                if i==0 :               # In order to have just one legend
                        # 3D plot
                        ax4.plot(pnt_plt[:, 0], pnt_plt[:, 1], pnt_plt[:, 2], color='black', label='Wing panels')
                        ax4.plot(vrt_plt[:, 0], vrt_plt[:, 1], vrt_plt[:, 2], color='red', lw=0.8, label='Vortex rings')
                else :
                        # 3D plot
                        ax4.plot(pnt_plt[:, 0], pnt_plt[:, 1], pnt_plt[:, 2], color='black')
                        ax4.plot(vrt_plt[:, 0], vrt_plt[:, 1], vrt_plt[:, 2], color='red', lw=0.8)
        for i, panel in enumerate(vlm.wake_panels):
                pnt    = copy.copy(panel.pnt)
                # Close the polygon shape
                pnt.append(pnt[0])
                pnt_plt = np.array(pnt)
                # Plot lattice
                if i==0 :               # In order to have just one legend
                        # 3D plot
                        ax4.plot(pnt_plt[:, 0], pnt_plt[:, 1], pnt_plt[:, 2], color='blue', lw=0.8, label='Wake panels')
                else :
                        # 3D plot
                        ax4.plot(pnt_plt[:, 0], pnt_plt[:, 1], pnt_plt[:, 2], color='blue', lw=0.8)      
        ax4.view_init(elev=11, azim=-148)
        x_lim = ax4.get_xlim3d()
        y_lim = ax4.get_ylim3d()
        z_lim = ax4.get_zlim3d()
        ax4.set_xlim(x_lim[0],x_lim[1])
        ax4.set_ylim(y_lim[1],y_lim[0]) 
        ax4.set_zlim(z_lim[1],z_lim[0])
        ax4.set_box_aspect([-x_lim[0]+x_lim[1],-y_lim[0]+y_lim[1],-z_lim[0]+z_lim[1]])
        ax4.set_xlabel("X [m]")
        ax4.set_ylabel("Y [m]")
        ax4.set_zlabel("Z [m]")
        ax4.yaxis.set_major_locator(MultipleLocator(0.1))
        ax4.set_title('3D view')
        ax4.legend()

        t2 = time.time()
        print("Computation time : ", t1-t0)
        print("Plot time        : ",t2-t1)

time_sim(b=B, u_inf=np.array([U, 0.0, 0.0]), n=N, t=T, dt=DT)